In [ ]:
# Load packages
import os
import tempfile
import subprocess

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

In [ ]:
# Create an isolated temporary Google Cloud SDK configuration directory
os.environ["CLOUDSDK_CONFIG"] = tempfile.mkdtemp(
        prefix="gcloud-dsub-",
        dir="/tmp",
)

# Print the active Cloud SDK configuration and HOME directories
print("CLOUDSDK_CONFIG:", os.environ["CLOUDSDK_CONFIG"])
print("CLOUDSDK_CONFIG =", os.environ.get("CLOUDSDK_CONFIG", "(default)"))
print("HOME =", os.environ.get("HOME"))

# Check the authenticated Google Cloud account
subprocess.run(
        ["gcloud", "config", "get-value", "account"],
        check=False,
)

# COMMAND 

In [ ]:
%%writefile convert_plink_to_bgen.sh

# Create a sample inclusion file from the FAM file
# Extract sample IDs (column 2) and use them as both sample identifiers
awk 'BEGIN{OFS="\t"} NR > 1 {print $2, $2}' "${FAM}" > samplelist.txt

# Preview variant and sample filtering files
head snplist.txt
head samplelist.txt

# Convert PLINK genotype data to BGEN format
plink2 \
    --bgen "${GENO}/chr${CHROM}.bgen" ref-last \
    --sample "${GENO}/chr${CHROM}.sample" \
    --extract "${BIM}" \
    --keep samplelist.txt \
    --export bgen-1.2 \
    --out "${OUTPUT}/chr${CHROM}"


# SUBMISSION

In [ ]:
import subprocess
import pandas as pd

chromosome = []
sizes = [] 

# Loop for all chromosome 
for chrom in range(1, 23):
    
    # Extract size of genomic PLINK BED file in GB for each chromosome [Version 8]
    command = [
        "gsutil",
        "-u", 'wb-cheerful-parsnip-892',
        "ls",
        "-lha",
        f"gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/bgen/chr{chrom}.bgen"
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    
    # Extract genomic size from string 
    lines = result.stdout.splitlines()
    line = lines[0].split()
    
    # Create list genomic size om is STILL undefined (critical failure)
    sizes.append(line[0])
    chromosome.append(chrom)
    
# Deposit genomic file size into data frame 
genomic_size = pd.DataFrame({
    "chromosome": chromosome,
    "size": sizes
})

print(genomic_size)

In [ ]:
# Define path to logging directory
LOGGING=f"{os.environ['WORKSPACE_BUCKET']}/data/logging"
# Define path to output directory 
OUTPUT=f"{os.environ['WORKSPACE_BUCKET']}/data/staar/bgen/"
# Define path to genomic data 
MNT_PATH="/mnt/data/input/gs/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/bgen"

MAX_DISK_SIZE = 500 # Set maximum disk; adjust for acaf or exome accordingly
EXTRA_DISK_SIZE = 20 # Set additional disk size for each chromosome 

# Loop through each chromosome 
for CHROM in [2, 4, 15, 17]:    
    
    print(f"Submitting dsub command for chromosome {CHROM}")
    
    DISK_SIZE = round(float(genomic_size.loc[genomic_size['chromosome'] == CHROM, 'size'].iloc[0])) + EXTRA_DISK_SIZE
    
    print(f"Requesting disk size: {DISK_SIZE}")

    if DISK_SIZE > MAX_DISK_SIZE: # Set maximum disk size 
        print(f"Disk too large for chromosome {CHROM}")
        break
        
    if DISK_SIZE > 1000: # Absolute maximum for version 8  
        print(f"Disk too large for chromosome {CHROM}")
        break

    # Define path to snp list 
    BIM=f"{os.environ['WORKSPACE_BUCKET']}/data/wgs_qc/wgs_variant_qc/exome/EUR_exome_chr{CHROM}_variant_qc.snplist"
    # Define path to sample list 
    FAM=f"{os.environ['WORKSPACE_BUCKET']}/data/wgs_qc/analysis_samples_wgs_qc/exome/EUR-like_exome_qc.txt"

    # Define path to genomic data [BGEN version 8]
    BGEN=f'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/bgen/chr{CHROM}.bgen'
    SAMPLE=f'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/bgen/chr{CHROM}.sample'
    BGI=f'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/bgen/chr{CHROM}.bgen.bgi'

    # Command for dsub submission  
    command = [
        "aou_dsub",
        "--project", os.getenv('GOOGLE_CLOUD_PROJECT'), 
        "--user-project", os.getenv('GOOGLE_CLOUD_PROJECT'), 
        "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/gfanz/plink2:latest",
        "--boot-disk-size", "10",
        "--disk-size", str(DISK_SIZE),
        "--machine-type", "n1-standard-1",
        "--logging", LOGGING,
        "--input", f"BIM={BIM}",
        "--input", f"FAM={FAM}",
        "--input", f"BGEN={BGEN}",
        "--input", f"SAMPLE={SAMPLE}",
        "--input", f"BGI={BGI}",
        "--output-recursive", f"OUTPUT={OUTPUT}",
        "--env", f"GENO={MNT_PATH}",
        "--env", f"CHROM={CHROM}",
        "--script", "convert_plink_to_bgen.sh"
    ]

    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {' '.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )

    print(dsub_command.stdout)
    print(dsub_command.stderr)